# Разбор упражнений к лекции 6

Условия — раздел 9 конспекта [`lecture06.md`](lecture06.md); краткие ответы без кода — [`exercises06.md`](exercises06.md). Все задачи домашние. **Сначала попробуйте сами**, потом сверяйтесь. Каждый разбор: условие с сутью → решение по пунктам → проверка кодом.

In [1]:
%matplotlib inline

import numpy as np
from l6helpers import *  # методы и задачи лекции (собираются из build_demo06.py)

## 6.0. Разминка — счёт руками

*Суть: Армихо, $\rho$, точка Коши и CG — на бумаге.*

**(а)** $\varphi(\alpha)=(1-2\alpha)^2$, $\varphi(0)=1$, $\varphi'(0)=-4$. Армихо с $c_1=\tfrac12$: $(1-2\alpha)^2\le1-2\alpha\iff4\alpha^2\le2\alpha\iff\alpha\le\tfrac12$. Backtracking: $\alpha=1$ даёт $\varphi=1>-1$ — отказ; $\alpha=\tfrac12$ даёт $\varphi=0\le0$ — принят (это точный минимум).

**(б)** $\rho=-1/4$: шаг отвергается, $\Delta\leftarrow\Delta/4$, подзадача решается заново из той же точки.

**(в)** $g^\top Bg=1$, $\lVert g\rVert=1$. При $\Delta=0.5$: $\tau=\min(1/(0.5\cdot1),1)=1$, $p_C=(-0.5,0)$ — на границе. При $\Delta=2$: $\tau=\min(1/2,1)=\tfrac12$, $p_C=(-1,0)$ — минимум вдоль $-g$ внутри шара.

**(г)** Два различных собственных числа — не больше $2$ итераций; пять различных — не больше $5$.

In [2]:
phi = lambda a: (1 - 2 * a) ** 2
print("6.0(а): backtracking", [(a, phi(a) <= 1 - 0.5 * a * 4) for a in (1, 0.5)])   # (α, Армихо выполнено?)
g, B = np.array([1.0, 0.0]), np.diag([1.0, 10.0])
print("6.0(в): точка Коши при Δ=0.5", cauchy_point(g, B, 0.5), " при Δ=2", cauchy_point(g, B, 2.0))
for A in (np.diag([1, 1, 5, 5, 5.0]), np.diag([1, 2, 3, 4, 5.0])):
    _, k, _ = cg(lambda v, A=A: A @ v, np.ones(5), np.zeros(5), tol=1e-12)
    print("6.0(г): CG итераций", k)

6.0(а): backtracking [(1, False), (0.5, True)]
6.0(в): точка Коши при Δ=0.5 [-0.5 -0. ]  при Δ=2 [-1. -0.]
6.0(г): CG итераций 2
6.0(г): CG итераций 5


## 6.1. Теорема о глобальной сходимости

*Суть: каждый шаг съедает кусок $\lVert g\rVert^2$, а съесть можно лишь конечное.*

**(а)** Доказательство целиком — четыре шага (Nocedal & Wright, теорема 3.2); $c_1$ — постоянная Армихо, backtracking делит шаг пополам.

*Шаг 1. Направление.* Собственные числа $B_k^{-1}$ лежат в $[1/M_B,\,1/m_B]$, поэтому для любого $v$: $\lVert v\rVert^2/M_B\le v^\top B_k^{-1}v\le\lVert v\rVert^2/m_B$. Для $p_k=-B_k^{-1}g_k$ отсюда $-g_k^\top p_k=g_k^\top B_k^{-1}g_k\ge\lVert g_k\rVert^2/M_B$ и $\lVert p_k\rVert\le\lVert g_k\rVert/m_B$; поделив квадрат первого неравенства на квадрат второго — условие угла $(-g_k^\top p_k)^2/\lVert p_k\rVert^2\ge(m_B/M_B)^2\lVert g_k\rVert^2$.

*Шаг 2. Лемма о спуске и оценка шага.* $f(x_k+\alpha p_k)-f_k=\int_0^1\nabla f(x_k+t\alpha p_k)^\top\alpha p_k\,dt=\alpha g_k^\top p_k+\int_0^1\big(\nabla f(x_k+t\alpha p_k)-g_k\big)^\top\alpha p_k\,dt$, а последний интеграл по липшицевости не больше $\int_0^1Lt\alpha^2\lVert p_k\rVert^2\,dt=\tfrac L2\alpha^2\lVert p_k\rVert^2$. Значит, Армихо $f(x_k+\alpha p_k)\le f_k+c_1\alpha g_k^\top p_k$ заведомо выполнено, когда $\alpha g_k^\top p_k+\tfrac L2\alpha^2\lVert p_k\rVert^2\le c_1\alpha g_k^\top p_k$, то есть при $\alpha\le\bar\alpha_k=2(1-c_1)(-g_k^\top p_k)/(L\lVert p_k\rVert^2)$. Backtracking от $\alpha=1$ остановится не позже, чем шаг впервые окажется $\le\bar\alpha_k$; предыдущий кандидат был $>\bar\alpha_k$, значит $\alpha_k\ge\min(1,\bar\alpha_k/2)$.

*Шаг 3. Убывание порядка $\lVert g_k\rVert^2$.* По Армихо $f_k-f_{k+1}\ge c_1\alpha_k(-g_k^\top p_k)$. Если $\bar\alpha_k/2\ge1$, то $\alpha_k=1$ и $f_k-f_{k+1}\ge c_1(-g_k^\top p_k)\ge c_1\lVert g_k\rVert^2/M_B$. Иначе $\alpha_k\ge\bar\alpha_k/2$ и $f_k-f_{k+1}\ge\tfrac{c_1}{2}\bar\alpha_k(-g_k^\top p_k)=\tfrac{c_1(1-c_1)}{L}\cdot\tfrac{(-g_k^\top p_k)^2}{\lVert p_k\rVert^2}\ge\tfrac{c_1(1-c_1)}{L}\big(\tfrac{m_B}{M_B}\big)^2\lVert g_k\rVert^2$. В обоих случаях $f_k-f_{k+1}\ge C\lVert g_k\rVert^2$ с $C=\min\big\{c_1/M_B,\ c_1(1-c_1)(m_B/M_B)^2/L\big\}>0$.

*Шаг 4. Суммирование.* $C\sum_{k=0}^N\lVert g_k\rVert^2\le\sum_{k=0}^N(f_k-f_{k+1})=f_0-f_{N+1}\le f_0-\inf f$. Правая часть не зависит от $N$, значит, ряд $\sum_k\lVert g_k\rVert^2$ сходится, а члены сходящегося ряда стремятся к нулю: $\lVert g_k\rVert\to0$. $\blacksquare$

**(б)** Для спуска $B=I$, $m_B=M_B=1$: $\lVert\nabla f\rVert\to0$ из любого старта. На Розенброке это гарантирует сходимость к $(1,1)$ — единственной стационарной точке, — но ничего не говорит о скорости: $13\,756$ итераций.

## 6.2. Сломайте Ньютона: сходимость в седло

*Суть: глобальная сходимость — к стационарной точке, а не к минимуму.*

In [3]:
TRAP = (0.58, 2.15)
x, k, _ = newton_ls(him, him_grad, him_hess, TRAP)                   # (а) сдвиг tau I только когда шаг не спуск
print(f"(а) сдвиг только при подъёме: {k} итераций → {x.round(4)}, собственные числа гессиана {np.linalg.eigvalsh(him_hess(x)).round(2)} — седло")
xa, ka, _ = newton_ls(him, him_grad, him_hess, TRAP, shift="always")  # (б) сдвиг всякий раз, когда lambda_min <= 1e-3
print(f"(б) сдвиг «всегда»: {ka} итераций → {xa.round(4)}, собственные числа {np.linalg.eigvalsh(him_hess(xa)).round(2)} — минимум")

(а) сдвиг только при подъёме: 5 итераций → [0.0867 2.8843], собственные числа гессиана [-31.71  75.51] — седло
(б) сдвиг «всегда»: 6 итераций → [3. 2.], собственные числа [25.72 82.28] — минимум


**(б)** При сдвиге «всегда, когда $\lambda_{\min}\le10^{-3}$» матрица $B_k\succ0$ на каждом шаге. Рядом с седлом компонента шага вдоль собственного вектора с отрицательным $\lambda$ направлена *от* седла (вдоль него функция убывает), поэтому седло отталкивает. Без сдвига направление Ньютона и так спуск — и метод честно идёт в седловую точку модели.

## 6.3. Точка Коши — спуск в маскировке

*Суть: точка Коши даёт сходимость, но скорость спуска.*

**(а)** $m(-tg)=f-t\lVert g\rVert^2+\tfrac12t^2g^\top Bg$ при $0\le t\le\Delta/\lVert g\rVert$. Если $g^\top Bg\le0$, функция убывает — берём границу ($\tau=1$). Иначе минимум параболы $t^\ast=\lVert g\rVert^2/g^\top Bg$ — обрезаем границей: $\tau=\min\big(\lVert g\rVert^3/(\Delta\,g^\top Bg),\,1\big)$.

**(б)** Точка Коши — это спуск с точным шагом (Л4 §4.4), обрезанным шаром, — та же линейная скорость $\sim\kappa$.

In [4]:
for step in ["cauchy", "dogleg"]:
    x, k, _, h = trust_region(rosen, rosen_grad, rosen_hess, X0_ROSEN, step=step)
    print(f"{step}: {k} итераций")
x, k, _ = gd(rosen, rosen_grad, X0_ROSEN)
print(f"спуск с backtracking (Л4): {k} итераций")

cauchy: 14110 итераций
dogleg: 24 итераций
спуск с backtracking (Л4): 13756 итераций


## 6.4. Dogleg руками

*Суть: ломаная Пауэлла — компромисс между точкой Коши и точкой Ньютона.*

$g=(4,2)$, $B=\operatorname{diag}(2,1)$: точка Ньютона $p_B=-B^{-1}g=(-2,-2)$, $\lVert p_B\rVert=2.83$; минимум вдоль $-g$: $p_U=-\frac{g^\top g}{g^\top Bg}g=-\frac{20}{36}(4,2)=(-2.22,-1.11)$, $\lVert p_U\rVert=2.48$.

- $\Delta=1<\lVert p_U\rVert$: первое колено, $p=-\Delta\,g/\lVert g\rVert=(-0.89,-0.45)$.
- $\Delta=2.6$: между $\lVert p_U\rVert$ и $\lVert p_B\rVert$ — второе колено, $p=p_U+t(p_B-p_U)$ с $\lVert p\rVert=2.6$: $t\approx0.43$, $p\approx(-2.13,-1.50)$.
- $\Delta=3>\lVert p_B\rVert$: шаг Ньютона $(-2,-2)$ — так при любом $\Delta\ge2.83$.

In [5]:
g, B = np.array([4.0, 2.0]), np.diag([2.0, 1.0])
for D in (1.0, 2.6, 3.0):
    p = dogleg(g, B, D)
    print(f"Δ = {D}: dogleg {p.round(3)}, |p| = {np.linalg.norm(p):.3f}, модель {g @ p + 0.5 * p @ B @ p:.3f}")

Δ = 1.0: dogleg [-0.894 -0.447], |p| = 1.000, модель -3.572
Δ = 2.6: dogleg [-2.126 -1.497], |p| = 2.600, модель -5.858
Δ = 3.0: dogleg [-2. -2.], |p| = 2.828, модель -6.000


## 6.5. Левенберг–Марквардт = доверительная область

*Суть: $\lambda$ и $\Delta$ — две ручки одного механизма.*

$p(\lambda)=-\big(\tfrac1{1+\lambda},\ \tfrac1{100+\lambda}\big)$ — длины ниже. Каждому $\lambda$ соответствует $\Delta=\lVert p(\lambda)\rVert$, при котором этот шаг — точное решение подзадачи (условия Moré–Sorensen). Длина монотонно убывает: $\frac{d}{d\lambda}\lVert p(\lambda)\rVert^2=-2\sum_i\frac{g_i^2}{(\lambda_i+\lambda)^3}<0$, поэтому «уменьшить $\Delta$» и «увеличить $\lambda$» — одно и то же.

In [6]:
JTJ, JTr = np.diag([1.0, 100.0]), np.array([1.0, 1.0])
for lam in [0, 1, 10, 100]:
    p = -np.linalg.solve(JTJ + lam * np.eye(2), JTr)        # шаг Левенберга–Марквардта
    D = np.linalg.norm(p)
    print(f"λ = {lam:3d}: p = {p.round(4)}, Δ = |p| = {D:.4f}; точный шаг подзадачи при этом Δ: {tr_exact(JTr, JTJ, D).round(4)}")

λ =   0: p = [-1.   -0.01], Δ = |p| = 1.0000; точный шаг подзадачи при этом Δ: [-1.   -0.01]
λ =   1: p = [-0.5    -0.0099], Δ = |p| = 0.5001; точный шаг подзадачи при этом Δ: [-0.5    -0.0099]
λ =  10: p = [-0.0909 -0.0091], Δ = |p| = 0.0914; точный шаг подзадачи при этом Δ: [-0.0909 -0.0091]
λ = 100: p = [-0.0099 -0.005 ], Δ = |p| = 0.0111; точный шаг подзадачи при этом Δ: [-0.0099 -0.005 ]


## 6.6. CG руками на эллипсе

*Суть: два шага — и точный минимум.*

$r_0=(50,50)$, $d_0=-(50,50)$, $\alpha_0=\frac{5000}{127\,500}=0.0392$, $x_1=(48.04,\,-0.961)$; $r_1=(48.04,\,-48.04)$, $\beta_1=0.923$, $d_1=(-94.2,\ 1.88)$, $\alpha_1=0.510$, $x_2=(0,0)$. Сопряжённость: $d_1^\top Ad_0=0$.

In [7]:
A = np.diag([1.0, 50.0]); x = np.array([50.0, 1.0]); r = A @ x; d = -r; ds = [d.copy()]
for _ in range(2):
    a = (r @ r) / (d @ A @ d); x = x + a * d; rn = r + a * A @ d        # шаг и новая невязка
    d = -rn + (rn @ rn) / (r @ r) * d; r = rn; ds.append(d.copy())       # новое направление
    print("x =", x.round(10))
print("d1^T A d0 =", round(ds[1] @ A @ ds[0], 8))

x = [48.03921569 -0.96078431]
x = [0. 0.]
d1^T A d0 = -0.0
